# For1Sight V2.7 – Machine Learning Development
## Stage 2: Feature Engineering and Dataset Preparation

### Objective
Prepare a leakage-controlled, event-aware modelling dataset for the post-qualifying / pre-race winner prediction task.

### Agreed temporal design
- 2024: initial model training set
- 2025: validation and model selection
- 2026: final unseen test set
- Exclude only the two unmatched 2026 events identified in Stage 1: Bahrain Grand Prix in Malaysia and United States Grand Prix.

### Important
This notebook does not train models, write to HDFS, alter the original V2.6 Parquet file, or modify Streamlit. It prepares and audits local modelling data only. Run cells from top to bottom.

## 0. Environment and project path

Select the same **Python (For1Sight)** Jupyter kernel used in Stage 1.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\ayepd\Desktop\For1Sight Project")
DATA_PATH = PROJECT_ROOT / "for1sight_warehouse" / "prediction_training_v2_6.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH, engine="pyarrow")
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

Loaded: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\prediction_training_v2_6.parquet
Shape: (1354, 49)


## 1. Apply V2.7 event eligibility rules

In [2]:
def normalize_event_name(value):
    value = str(value or "").lower().strip()
    value = re.sub(r"\bgrand\s*prix\b", "", value)
    return re.sub(r"[^a-z0-9]", "", value)

def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})

required = [
    "year", "event_name", "driver_code", "constructor",
    "finishing_position", "is_winner",
    "event_prediction_ready", "model_training_eligible",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required V2.6 columns are missing: {missing}")

work = df.copy()
work["year"] = pd.to_numeric(work["year"], errors="coerce").astype("Int64")
work["event_key"] = work["event_name"].map(normalize_event_name)
work["target"] = pd.to_numeric(work["is_winner"], errors="coerce")
work["finishing_position_check"] = pd.to_numeric(work["finishing_position"], errors="coerce")
work["event_prediction_ready_check"] = to_bool(work["event_prediction_ready"])
work["model_training_eligible_check"] = to_bool(work["model_training_eligible"])

excluded_2026_keys = {"bahraininmalaysia", "unitedstates"}
excluded_mask = (work["year"] == 2026) & work["event_key"].isin(excluded_2026_keys)

eligible = work.loc[
    ~excluded_mask
    & work["event_prediction_ready_check"]
    & work["model_training_eligible_check"]
    & work["target"].isin([0, 1])
    & work["finishing_position_check"].notna()
].copy()

eligible["event_id"] = (
    eligible["year"].astype(str) + "_" + eligible["event_key"].astype(str)
)

print("Original rows:", len(work))
print("Excluded rows (two unmatched 2026 events):", int(excluded_mask.sum()))
print("Eligible rows:", len(eligible))
print("Eligible events:", eligible["event_id"].nunique())
display(
    eligible.groupby("year").agg(
        rows=("event_id", "size"),
        events=("event_id", "nunique"),
        winner_labels=("target", "sum")
    ).reset_index().sort_values("year")
)

Original rows: 1354
Excluded rows (two unmatched 2026 events): 44
Eligible rows: 1310
Eligible events: 64


,year,rows,events,winner_labels
0,2024,479,24,24
1,2025,479,24,24
2,2026,352,16,16


## 2. Validate event-level target integrity

In [3]:
event_integrity = (
    eligible.groupby(["year", "event_id", "event_name"])
    .agg(
        driver_rows=("driver_code", "size"),
        winner_labels=("target", "sum"),
        distinct_drivers=("driver_code", "nunique"),
    )
    .reset_index()
)
event_integrity["valid_event"] = (
    (event_integrity["winner_labels"] == 1)
    & (event_integrity["driver_rows"] == event_integrity["distinct_drivers"])
)

print("Events passing integrity check:",
      int(event_integrity["valid_event"].sum()), "/", len(event_integrity))
display(event_integrity.loc[~event_integrity["valid_event"]])

if not event_integrity["valid_event"].all():
    raise ValueError("Some events have duplicate drivers or do not have exactly one winner.")

Events passing integrity check: 64 / 64


,year,event_id,event_name,driver_rows,winner_labels,distinct_drivers,valid_event


## 3. Define the prediction features

The model predicts `target` (whether a driver wins the event).

The following are candidate inputs available after qualifying:
- Grid position and qualifying pace
- Practice pace and timed-lap coverage
- Driver's previous performance
- Constructor's previous performance
- Previous driver / constructor performance at the selected circuit
- Driver, constructor and circuit identity

We deliberately exclude post-race results and data-quality summary fields from model inputs.

In [4]:
NUMERIC_FEATURES = [
    "round",
    "starting_grid",
    "best_practice_lap_s",
    "best_qualifying_lap_s",
    "best_quali_s1_s",
    "best_quali_s2_s",
    "best_quali_s3_s",
    "theoretical_best_sector_sum_s",
    "practice_to_qualifying_improvement_s",
    "practice_delta_to_event_best_s",
    "qualifying_delta_to_event_best_s",
    "event_best_practice_lap_s",
    "event_best_qualifying_lap_s",
    "qualifying_rank",
    "practice_timed_laps",
    "qualifying_timed_laps",
    "driver_prior_avg_finish_3",
    "driver_prior_avg_points_3",
    "driver_circuit_prior_avg_finish",
    "driver_circuit_prior_starts",
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
    "has_practice_data",
    "has_qualifying_data",
    "has_grid_data",
]

CATEGORICAL_FEATURES = [
    "driver_code_norm",
    "constructor",
    "event_norm",
]

TARGET_COLUMN = "target"
GROUP_COLUMN = "event_id"

missing_features = [
    c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES
    if c not in eligible.columns
]
print("Missing proposed features:", missing_features if missing_features else "None")

if missing_features:
    raise ValueError("Review feature list against actual V2.6 schema before proceeding.")

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total candidate features:", len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES))

Missing proposed features: None
Numeric features: 27
Categorical features: 3
Total candidate features: 30


## 4. Feature missingness and treatment plan

In [5]:
feature_missingness = pd.DataFrame({
    "feature": NUMERIC_FEATURES + CATEGORICAL_FEATURES,
    "missing_count": [int(eligible[c].isna().sum()) for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES],
    "missing_percent": [
        round(float(eligible[c].isna().mean() * 100), 2)
        for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES
    ],
}).sort_values("missing_percent", ascending=False)

display(feature_missingness)

print("Proposed treatment:")
print("- Numeric features: median imputation with missingness indicators.")
print("- Categorical features: most-frequent imputation and one-hot encoding.")
print("- Circuit prior average features remain candidates; their missingness will be explicitly flagged.")

,feature,missing_count,missing_percent
18,driver_circuit_prior_avg_finish,648,49.47
22,constructor_circuit_prior_avg_finish,561,42.82
17,driver_prior_avg_points_3,28,2.14
16,driver_prior_avg_finish_3,28,2.14
20,constructor_prior_avg_finish_3,24,1.83
21,constructor_prior_avg_points_3,24,1.83
8,practice_to_qualifying_improvement_s,10,0.76
10,qualifying_delta_to_event_best_s,9,0.69
13,qualifying_rank,9,0.69
4,best_quali_s1_s,9,0.69


Proposed treatment:
- Numeric features: median imputation with missingness indicators.
- Categorical features: most-frequent imputation and one-hot encoding.
- Circuit prior average features remain candidates; their missingness will be explicitly flagged.


## 5. Check numeric ranges and potential anomalies

In [6]:
numeric_summary = eligible[NUMERIC_FEATURES].describe().T
display(numeric_summary)

if "starting_grid" in eligible.columns:
    print("Starting grid values:")
    display(eligible["starting_grid"].value_counts(dropna=False).sort_index())

if "qualifying_rank" in eligible.columns:
    print("Qualifying rank values:")
    display(eligible["qualifying_rank"].value_counts(dropna=False).sort_index())

,count,mean,std,min,25%,50%,75%,max
round,1310.0,11.451908,6.641909,1.000,6.000000,11.000000,17.000000,24.000000
starting_grid,1309.0,10.747135,5.936031,1.000,6.000000,11.000000,16.000000,22.000000
best_practice_lap_s,1309.0,84.430523,10.616817,64.324,75.700000,83.722000,92.047000,110.631000
best_qualifying_lap_s,1301.0,84.372341,11.437945,63.971,75.281000,83.468000,91.496000,121.101000
best_quali_s1_s,1301.0,26.141487,4.990443,16.251,23.408000,26.841000,29.319000,41.429000
best_quali_s2_s,1301.0,32.345587,7.136780,16.915,27.446000,31.184000,37.663000,54.494000
best_quali_s3_s,1301.0,25.805689,5.942173,15.941,21.521000,24.995000,29.148000,45.688000
theoretical_best_sector_sum_s,1301.0,84.292763,11.407792,63.971,75.249000,83.438000,91.394000,121.101000
practice_to_qualifying_improvement_s,1300.0,0.059103,3.785239,-31.324,0.410750,0.814500,1.212750,4.788000
practice_delta_to_event_best_s,1309.0,1.156870,0.888157,0.000,0.566000,1.004000,1.519000,6.763000


Starting grid values:


starting_grid
1.0     64
2.0     64
3.0     64
4.0     64
5.0     64
6.0     64
7.0     64
8.0     64
9.0     64
10.0    64
11.0    64
12.0    64
13.0    64
14.0    64
15.0    64
16.0    64
17.0    64
18.0    64
19.0    64
20.0    61
21.0    16
22.0    16
NaN      1
Name: count, dtype: int64

Qualifying rank values:


qualifying_rank
1.0     64
2.0     65
3.0     65
4.0     64
5.0     64
6.0     64
7.0     65
8.0     64
9.0     64
10.0    64
11.0    64
12.0    65
13.0    64
14.0    64
15.0    64
16.0    64
17.0    64
18.0    64
19.0    63
20.0    54
21.0    15
22.0    13
NaN      9
Name: count, dtype: int64

## 6. Check categorical coverage and unseen categories

The 2025 and 2026 seasons may contain drivers or teams not seen in the 2024 training season. The planned OneHotEncoder will use `handle_unknown='ignore'` so unseen categories do not crash inference.

In [7]:
for col in CATEGORICAL_FEATURES:
    print(f"\\n{col}")
    coverage = (
        eligible.groupby("year")[col]
        .agg(lambda s: sorted(s.dropna().astype(str).unique().tolist()))
    )
    for season, values in coverage.items():
        print(season, ":", len(values), "unique values")
    train_values = set(eligible.loc[eligible["year"] == 2024, col].dropna().astype(str))
    for season in [2025, 2026]:
        season_values = set(eligible.loc[eligible["year"] == season, col].dropna().astype(str))
        print(f"Unseen in 2024 but present in {season}:", sorted(season_values - train_values))

\ndriver_code_norm
2024 : 24 unique values
2025 : 21 unique values
2026 : 23 unique values
Unseen in 2024 but present in 2025: ['ANT', 'BOR', 'HAD']
Unseen in 2024 but present in 2026: ['ANT', 'BOR', 'HAD', 'LIN']
\nconstructor
2024 : 10 unique values
2025 : 10 unique values
2026 : 11 unique values
Unseen in 2024 but present in 2025: []
Unseen in 2024 but present in 2026: ['Audi', 'Cadillac F1 Team']
\nevent_norm
2024 : 24 unique values
2025 : 24 unique values
2026 : 16 unique values
Unseen in 2024 but present in 2025: []
Unseen in 2024 but present in 2026: ['barcelona']


## 7. Create chronological train / validation / test datasets

In [8]:
train_df = eligible.loc[eligible["year"] == 2024].copy()
valid_df = eligible.loc[eligible["year"] == 2025].copy()
test_df  = eligible.loc[eligible["year"] == 2026].copy()

split_summary = pd.DataFrame([
    {
        "split": "Train",
        "season": 2024,
        "rows": len(train_df),
        "events": train_df[GROUP_COLUMN].nunique(),
        "winner_rows": int(train_df[TARGET_COLUMN].sum()),
    },
    {
        "split": "Validation",
        "season": 2025,
        "rows": len(valid_df),
        "events": valid_df[GROUP_COLUMN].nunique(),
        "winner_rows": int(valid_df[TARGET_COLUMN].sum()),
    },
    {
        "split": "Final test",
        "season": 2026,
        "rows": len(test_df),
        "events": test_df[GROUP_COLUMN].nunique(),
        "winner_rows": int(test_df[TARGET_COLUMN].sum()),
    },
])
display(split_summary)

if train_df.empty or valid_df.empty or test_df.empty:
    raise ValueError("One or more chronological splits are empty.")

,split,season,rows,events,winner_rows
0,Train,2024,479,24,24
1,Validation,2025,479,24,24
2,Final test,2026,352,16,16


## 8. Build X/y tables (raw features only)

No imputation, scaling or one-hot encoding is fitted here. Those transformations must be fitted on the training set only inside a Scikit-learn Pipeline to avoid data leakage.

In [9]:
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].astype(int).copy()

X_valid = valid_df[FEATURE_COLUMNS].copy()
y_valid = valid_df[TARGET_COLUMN].astype(int).copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].astype(int).copy()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "| y_valid:", y_valid.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)

display(X_train.head())

X_train: (479, 30) | y_train: (479,)
X_valid: (479, 30) | y_valid: (479,)
X_test : (352, 30) | y_test : (352,)


,round,starting_grid,best_practice_lap_s,best_qualifying_lap_s,best_quali_s1_s,best_quali_s2_s,best_quali_s3_s,theoretical_best_sector_sum_s,practice_to_qualifying_improvement_s,practice_delta_to_event_best_s,...,constructor_prior_avg_finish_3,constructor_prior_avg_points_3,constructor_circuit_prior_avg_finish,constructor_circuit_prior_starts,has_practice_data,has_qualifying_data,has_grid_data,driver_code_norm,constructor,event_norm
0,1,1.0,90.851,89.179,28.509,38.269,22.374,89.152,1.672,0.477,...,NaN,NaN,NaN,0,True,True,True,VER,Red Bull,bahrain
1,1,2.0,91.094,89.165,28.762,38.059,22.263,89.084,1.929,0.720,...,NaN,NaN,NaN,0,True,True,True,LEC,Ferrari,bahrain
2,1,3.0,90.580,89.485,28.800,38.421,22.264,89.485,1.095,0.206,...,NaN,NaN,NaN,0,True,True,True,RUS,Mercedes,bahrain
3,1,4.0,90.769,89.507,28.850,38.298,22.337,89.485,1.262,0.395,...,NaN,NaN,NaN,0,True,True,True,SAI,Ferrari,bahrain
4,1,5.0,91.115,89.537,28.738,38.323,22.410,89.471,1.578,0.741,...,NaN,NaN,NaN,0,True,True,True,PER,Red Bull,bahrain


## 9. Establish the starting-grid baseline

Before judging machine-learning models, we need a simple baseline: predict the driver starting from P1 as the winner.

We also calculate Top-3 coverage: whether the actual winner started in the first three grid positions. This is a useful race-level ranking benchmark.

In [10]:
def starting_grid_baseline(frame, split_name):
    records = []
    for event_id, group in frame.groupby(GROUP_COLUMN):
        valid_grid = group.dropna(subset=["starting_grid"]).copy()
        if valid_grid.empty:
            continue
        valid_grid["grid_rank"] = valid_grid["starting_grid"].rank(method="min", ascending=True)
        winner_rows = valid_grid.loc[valid_grid[TARGET_COLUMN] == 1]
        if winner_rows.empty:
            continue
        winner_grid_rank = float(winner_rows.iloc[0]["grid_rank"])
        predicted = valid_grid.loc[valid_grid["starting_grid"] == valid_grid["starting_grid"].min()]
        predicted_winner = bool(predicted[TARGET_COLUMN].eq(1).any())
        records.append({
            "event_id": event_id,
            "actual_winner_grid_rank": winner_grid_rank,
            "top1_correct": predicted_winner,
            "top3_correct": winner_grid_rank <= 3,
        })
    result = pd.DataFrame(records)
    return {
        "split": split_name,
        "events_evaluated": len(result),
        "top1_accuracy_pct": round(result["top1_correct"].mean() * 100, 2) if len(result) else np.nan,
        "top3_accuracy_pct": round(result["top3_correct"].mean() * 100, 2) if len(result) else np.nan,
        "details": result,
    }

baseline_valid = starting_grid_baseline(valid_df, "2025 validation")
baseline_test = starting_grid_baseline(test_df, "2026 final test")

baseline_summary = pd.DataFrame([
    {k: v for k, v in baseline_valid.items() if k != "details"},
    {k: v for k, v in baseline_test.items() if k != "details"},
])
display(baseline_summary)

print("2025 event-level baseline details:")
display(baseline_valid["details"])
print("2026 event-level baseline details:")
display(baseline_test["details"])

,split,events_evaluated,top1_accuracy_pct,top3_accuracy_pct
0,2025 validation,24,66.67,95.83
1,2026 final test,16,68.75,93.75


2025 event-level baseline details:


,event_id,actual_winner_grid_rank,top1_correct,top3_correct
0,2025_abudhabi,1.0,True,True
1,2025_australian,1.0,True,True
2,2025_austrian,1.0,True,True
3,2025_azerbaijan,1.0,True,True
4,2025_bahrain,1.0,True,True
5,2025_belgian,2.0,False,True
6,2025_british,3.0,False,True
7,2025_canadian,1.0,True,True
8,2025_chinese,1.0,True,True
9,2025_dutch,1.0,True,True


2026 event-level baseline details:


,event_id,actual_winner_grid_rank,top1_correct,top3_correct
0,2026_australian,1.0,True,True
1,2026_austrian,1.0,True,True
2,2026_azerbaijan,1.0,True,True
3,2026_barcelona,2.0,False,True
4,2026_belgian,1.0,True,True
5,2026_british,2.0,False,True
6,2026_canadian,2.0,False,True
7,2026_chinese,1.0,True,True
8,2026_dutch,1.0,True,True
9,2026_hungarian,1.0,True,True


## 10. Feature leakage audit

This explicit audit confirms that outcome fields are not included in the model input list.

In [11]:
FORBIDDEN_INPUTS = {
    "target", "is_winner", "finishing_position", "finishing_position_check",
    "race_points", "race_status", "winner_label_count",
    "event_prediction_ready", "model_training_eligible",
    "event_driver_count", "qualifying_feature_count", "grid_feature_count",
    "qualifying_coverage", "grid_coverage", "feature_match_status",
    "event_prediction_ready_check", "model_training_eligible_check",
}

leakage_hits = sorted(set(FEATURE_COLUMNS) & FORBIDDEN_INPUTS)
print("Forbidden fields included in features:", leakage_hits if leakage_hits else "None")
assert not leakage_hits, "Leakage fields found in model features."

print("Leakage audit passed.")

Forbidden fields included in features: None
Leakage audit passed.


## 11. Stage 2 completion summary

Before moving to Stage 3, confirm:
- [ ] Eligible rows remain 1,310 across 64 events.
- [ ] Season split is 2024 / 2025 / 2026 with 24 / 24 / 16 events.
- [ ] Every eligible event has exactly one winner and no duplicate driver rows.
- [ ] The feature list contains only pre-race / post-qualifying information.
- [ ] Numeric missingness will be handled inside the training pipeline.
- [ ] Categorical unseen values are handled with `OneHotEncoder(handle_unknown='ignore')`.
- [ ] Starting-grid baseline has been calculated for 2025 and 2026.
- [ ] Leakage audit passes.

**Next stage:** Stage 3 – train and compare Logistic Regression, Random Forest and Gradient Boosting. All preprocessing will be fitted on the training data only. Model selection will use 2025 validation results; 2026 remains untouched until final evaluation.